# Limpieza de Datos (Data cleaning) - Proyecto Banca
# Equipo 30 - 08/06/2026


In [1]:
import pandas as pd
import numpy as np

# Configuración de visualización
pd.set_option('display.max_columns', None)


---
## 1. Carga y Preparación Inicial
---

In [ ]:
# Cargar del dataset
df = pd.read_csv("../../Data/06-08-2026/06-08-2026_Raw.csv", index_col=False, encoding='utf-8')

print(f"Dimensiones iniciales: {df.shape}")


Dimensiones iniciales: (11162, 18)


---
## 2. Comprobacion de Duplicados
---

* 1. Comprobar duplicados por ID único
* 2. Comprobar si hay filas exactamente iguales (sin contar el ID)
* 3. Limpieza (en caso de que existan, elimina manteniendo la primera aparición)

In [17]:
duplicados_id = df.duplicated(subset=['id']).sum()
print(f"Filas con ID duplicado: {duplicados_id}")


columnas_sin_id = [col for col in df.columns if col != 'id']
duplicados_filas = df.duplicated(subset=columnas_sin_id).sum()
print(f"Filas duplicadas en contenido (ignorando ID): {duplicados_filas}")


if duplicados_filas > 0:
    df = df.drop_duplicates(subset=columnas_sin_id, keep='first')

Filas con ID duplicado: 0
Filas duplicadas en contenido (ignorando ID): 0


In [18]:
df.shape

(11162, 18)

In [19]:
# Eliminación de posibles duplicados
df = df.drop_duplicates()
print(f"Dimensiones tras eliminar duplicados: {df.shape}")


Dimensiones tras eliminar duplicados: (11162, 18)


Mostrar el dataset

In [20]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59.0,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,2,56.0,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,3,41.0,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,4,55.0,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,5,54.0,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,11158,33.0,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,no
11158,11159,39.0,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,no
11159,11160,32.0,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,no
11160,11161,43.0,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,no


---
## 3. Tratamiento de Valores Faltantes
---

### Valores faltantes | valores nulos por columna

In [21]:
# Comprobar si hay valores faltantes
missing_values = df.isnull().sum() | df.isna().sum()

In [22]:
missing_values

id            0
age          10
job           0
marital       5
education     7
default       0
balance       0
housing       0
loan          0
contact       0
day           0
month         0
duration      0
campaign      0
pdays         0
previous      0
poutcome      0
deposit       0
dtype: int64

* 1. Imputar numéricos ('age') con la mediana teniendo en cuenta los valores de las demás variables ('education', 'job').
* 2. Imputar categóricos ('marital', 'education', 'job') con la moda (para los nulls de marital y education, y los unknown de education y job) teniendo en cuenta los valores de las demás variables.
* 3. Manejo de "unknown" estratégicos
    * 'contact' (20%) se quedan como "unknown" porque ya es una categoría.
    * Renombrar el "unknown" de 'poutcome' a su significado de negocio 'no_pcampaign'.

In [ ]:
# Imputación de valores faltantes y renombrado de categorías

def get_mode(series):
    """Obtiene la moda gestionando grupos vacíos."""
    mode = series.mode()
    return mode[0] if not mode.empty else 'unknown'

# Grupos de edad temporales para imputación
bins = [17, 24, 34, 49, 64, 95]
labels = ['18-24', '25-34', '35-49', '50-64', '65+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)


# Imputación de Edad (age)
# Basado en educación y trabajo
df['age'] = df.groupby(['education', 'job'], observed=True)['age'].transform(
    lambda x: x.fillna(x.median())
)
df['age'] = df.groupby('job', observed=True)['age'].transform(
    lambda x: x.fillna(x.median())
)
df['age'] = df.groupby('education', observed=True)['age'].transform(
    lambda x: x.fillna(x.median())
)
df['age'] = df['age'].fillna(df['age'].median())


# Imputación de Educación (education)
# Basado en trabajo y grupo de edad
df['education'] = df.groupby(['job', 'age_group'], observed=True)['education']\
    .transform(lambda x: x.fillna(get_mode(x)))

df['education'] = df.groupby('job', observed=True)['education'].transform(
    lambda x: x.fillna(get_mode(x))
)
df['education'] = df.groupby('age_group', observed=True)['education'].transform(
    lambda x: x.fillna(get_mode(x))
)
df['education'] = df['education'].fillna(get_mode(df['education']))


# Imputación de Estado Civil (marital)
# Basado en grupo de edad y trabajo
df['marital'] = df.groupby(['age_group', 'job'], observed=True)['marital']\
    .transform(lambda x: x.fillna(get_mode(x)))

df['marital'] = df.groupby('age_group', observed=True)['marital'].transform(
    lambda x: x.fillna(get_mode(x))
)
df['marital'] = df.groupby('job', observed=True)['marital'].transform(
    lambda x: x.fillna(get_mode(x))
)
df['marital'] = df['marital'].fillna(get_mode(df['marital']))


# Renombrar 'unknown' en poutcome a 'no_campaign'
df['poutcome'] = df['poutcome'].replace('unknown', 'no_campaign')


# Eliminar columna auxiliar
df.drop('age_group', axis=1, inplace=True)

print("Nulos tras imputación:\n", df.isnull().sum())
print("\nValores únicos en poutcome:", df['poutcome'].unique())



Nulos tras imputación:
 id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64

Valores únicos en poutcome: ['no_campaign' 'other' 'failure' 'success']


---
## 4. Estandarización de formatos y tipos de datos
---

* 1. Estandarizar textos a minúsculas y sin espacios
* 2. Mapear todas las variables binarias de yes/no a 1/0 enteros (dejamos el dataset nativamente listo para cualquier algoritmo estadístico sin necesidad de hacer pasos extra más adelante.)
* 3. Corrección de tipos numéricos (asegurar enteros)
* 4. Convertir columnas categóricas multi-clase a tipo 'category' (las columnas de texto (object) consumen mucha memoria porque guardan cada palabra textualmente. El tipo category funciona como un diccionario: asigna un número interno (ej. Married = 1, Single = 2) pero nos sigue mostrando el texto)

In [24]:
columnas_texto = df.select_dtypes(include=['object']).columns
for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip().str.lower()


columnas_binarias = ['deposit', 'default', 'housing', 'loan']
for col in columnas_binarias:
    df[col] = df[col].map({'yes': 1, 'no': 0})


df['age'] = df['age'].astype(int)

columnas_a_categoria = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
for col in columnas_a_categoria:
    if col in df.columns:
        df[col] = df[col].astype('category')

---
## 5.Verificación final del estado del Dataframe
---

In [25]:
print(f"Final Dataframe Shape: {df.shape}")


Final Dataframe Shape: (11162, 18)


In [26]:
print("\nMissing values check:")
print(df.isnull().sum())


Missing values check:
id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64


In [27]:
print("\nOptimized Data Types:")
print(df.dtypes)


Optimized Data Types:
id              int64
age             int64
job          category
marital      category
education    category
default         int64
balance         int64
housing         int64
loan            int64
contact      category
day             int64
month        category
duration        int64
campaign        int64
pdays           int64
previous        int64
poutcome     category
deposit         int64
dtype: object


In [28]:
print("\nUnique values in poutcome:")
print(df['poutcome'].unique())


Unique values in poutcome:
['no_campaign', 'other', 'failure', 'success']
Categories (4, object): ['failure', 'no_campaign', 'other', 'success']


Mostrar el dataset

In [30]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_campaign,1
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_campaign,1
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_campaign,1
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_campaign,1
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_campaign,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,11158,33,blue-collar,single,primary,0,1,1,0,cellular,20,apr,257,1,-1,0,no_campaign,0
11158,11159,39,services,married,secondary,0,733,0,0,unknown,16,jun,83,4,-1,0,no_campaign,0
11159,11160,32,technician,single,secondary,0,29,0,0,cellular,19,aug,156,2,-1,0,no_campaign,0
11160,11161,43,technician,married,secondary,0,0,0,1,cellular,8,may,9,2,172,5,failure,0


In [29]:
# Guardado del dataset limpio
df.to_csv("../../Data/06-08-2026/06-08-2026_Clean.csv", index=False)

print(f"\nArchivo guardado")


Archivo guardado
